In [1]:
%run ./nb_ingest_silver_acto_gestao_obras_santos

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 5, Finished, Available, Finished, True)

Solicitações: 10984
OSs tempo/etapa: 10983
Tabela de solicitações: 10984 linhas
Tabela de etapas: 84022 linhas


In [2]:
import pandas as pd
# ============================================================
# Carga das tabelas Silver – Acto Obras Santos
# ============================================================

df_obras_etapas = pd.read_parquet(
    "/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_etapas.parquet"
)

df_obras_solicitacoes = pd.read_parquet(
    "/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_solicitacoes.parquet"
)

print(f"✓ Etapas carregadas: {len(df_obras_etapas):,} registros")
print(f"✓ Solicitações carregadas: {len(df_obras_solicitacoes):,} registros")

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 6, Finished, Available, Finished, False)

✓ Etapas carregadas: 84,022 registros
✓ Solicitações carregadas: 10,984 registros


In [3]:
df_obras_solicitacoes

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 7, Finished, Available, Finished, False)

,Nº Solicitação|1,Status Fluxo|3,Data Finalização|4,Bairro|25,Nome do Logradouro|81,Número|90,Nº Solicitação|2,Status Fluxo|6,Data Finalização|7,Nome do Logradouro|52,...,Número Processo|8,Status Etapa|10,Etapa|11,Executor Etapa|14,Solicitante|6,Data Execução Etapa|13,Data Criação Etapa|12,Protocolo Cliente Ano|9,Beneficiário|7,Data Criação|5
0,833625,Em atendimento,2025-09-15T18:23:29.052Z,,,,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,835351,Pendente atendimento,2025-09-18T18:02:12.449Z,,,,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,843923,Em atendimento,2025-12-08T14:25:35.256Z,POMPEIA,BENEDITO CALIXTO,1,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,846680,Em atendimento,2025-10-13T18:50:32.289Z,,,,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,847145,Em atendimento,2025-10-14T18:06:41.215Z,,,,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10979,931140,Em atendimento,2026-04-06T14:40:59.658Z,None,None,None,None,None,None,None,...,,Finalizado,DADOS DO IMÓVEL E DO PROJETO,RENATA DIAS MANZANO,RENATA DIAS MANZANO,2026-03-31T13:38:29Z,2026-03-31T13:38:26Z,901541/2026-51,,2026-03-31T13:38:26Z
10980,931790,Em atendimento,2026-04-09T10:59:21.691Z,None,None,None,None,None,None,None,...,,Finalizado,DADOS DO IMÓVEL E DO PROJETO,MARCOS SIMOES ELIAS,MARCOS SIMOES ELIAS,2026-04-01T11:39:03Z,2026-04-01T11:39:00Z,901569/2026-70,,2026-04-01T11:39:00Z
10981,932945,Em atendimento,2026-04-07T19:39:08.257Z,None,None,None,None,None,None,None,...,,Pendente atendimento,DADOS DO IMÓVEL E DO PROJETO,,FREDERICO DA COSTA MARINS,,2026-04-07T19:39:06Z,901599/2026-31,,2026-04-02T18:42:47Z
10982,933089,Em atendimento,2026-04-09T12:36:23.042Z,None,None,None,None,None,None,None,...,,Finalizado,DADOS DO IMÓVEL E DO PROJETO,RENATA DIAS MANZANO,RENATA DIAS MANZANO,2026-04-04T15:20:09Z,2026-04-04T15:19:12Z,901603/2026-14,,2026-04-04T15:19:05Z


In [4]:
# ============================================================
# Tratamento – BFill horizontal nas SOLICITAÇÕES
# ============================================================

def aplicar_bfill(df, coluna: str):
    cols_selecao = df.filter(like=coluna).columns.tolist()

    if not cols_selecao:
        return df

    df[coluna] = df[cols_selecao].bfill(axis=1).iloc[:, 0]
    df = df.drop(columns=cols_selecao)

    return df


# Colunas finais desejadas
COLUNAS_FINAIS = [
    "Nº Solicitação",
    "Serviço",
    "Status Fluxo",
    "Data Criação",
    "Data Finalização",
    "Data Criação Etapa",
    "Solicitante",
    "Bairro",
    "Executor",
    "Etapa",
    "Título Profissional",
]

df_solicitacoes_tratado = df_obras_solicitacoes.copy()

# Aplica bfill
for coluna in COLUNAS_FINAIS:
    df_solicitacoes_tratado = aplicar_bfill(df_solicitacoes_tratado, coluna)

#DROP FINAL — mantém somente o que interessa
df_solicitacoes_tratado = df_solicitacoes_tratado[
    [c for c in COLUNAS_FINAIS if c in df_solicitacoes_tratado.columns]
]

print("✓ BFill aplicado e colunas normalizadas")
print(f"✓ Registros: {len(df_solicitacoes_tratado):,}")
print(f"✓ Colunas finais: {df_solicitacoes_tratado.columns.tolist()}")


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 8, Finished, Available, Finished, False)

✓ BFill aplicado e colunas normalizadas
✓ Registros: 10,984
✓ Colunas finais: ['Nº Solicitação', 'Serviço', 'Status Fluxo', 'Data Criação', 'Data Finalização', 'Solicitante', 'Bairro', 'Executor', 'Etapa', 'Título Profissional']


In [5]:
df_solicitacoes_tratado

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 9, Finished, Available, Finished, False)

,Nº Solicitação,Serviço,Status Fluxo,Data Criação,Data Finalização,Solicitante,Bairro,Executor,Etapa,Título Profissional
0,833625,None,Em atendimento,None,2025-09-15T18:23:29.052Z,None,,None,None,None
1,835351,None,Pendente atendimento,None,2025-09-18T18:02:12.449Z,None,,None,None,None
2,843923,None,Em atendimento,None,2025-12-08T14:25:35.256Z,None,POMPEIA,None,None,None
3,846680,None,Em atendimento,None,2025-10-13T18:50:32.289Z,None,,None,None,None
4,847145,None,Em atendimento,None,2025-10-14T18:06:41.215Z,None,,None,None,None
...,...,...,...,...,...,...,...,...,...,...
10979,931140,REFORMA E/OU LEGALIZAÇÃO,Em atendimento,2026-03-31T13:38:26Z,2026-04-06T14:40:59.658Z,RENATA DIAS MANZANO,None,RENATA DIAS MANZANO,Finalizado,None
10980,931790,REFORMA E/OU LEGALIZAÇÃO,Em atendimento,2026-04-01T11:39:00Z,2026-04-09T10:59:21.691Z,MARCOS SIMOES ELIAS,None,MARCOS SIMOES ELIAS,Finalizado,None
10981,932945,REFORMA E/OU LEGALIZAÇÃO,Em atendimento,2026-04-07T19:39:06Z,2026-04-07T19:39:08.257Z,FREDERICO DA COSTA MARINS,None,,Pendente atendimento,None
10982,933089,REFORMA E/OU LEGALIZAÇÃO,Em atendimento,2026-04-04T15:19:12Z,2026-04-09T12:36:23.042Z,RENATA DIAS MANZANO,None,RENATA DIAS MANZANO,Finalizado,None


In [6]:
#TESTE VERIFICAÇÃO  
import pandas as pd

df = df_solicitacoes_tratado.copy()

def adicionar_etapa_atual_2(
    df_etapas: pd.DataFrame,
    df_solicitacoes: pd.DataFrame
) -> pd.DataFrame:

    # ============================================================
    # 1. Preparação das etapas
    # ============================================================
    df_etapas = df_etapas.copy()
    df_etapas["seqFluxo"] = df_etapas["seqFluxo"].astype("int64")
    df_etapas["dataAtenderEtapa"] = pd.to_datetime(df_etapas["dataAtenderEtapa"], errors="coerce")

    if "dataEtapaFim" in df_etapas.columns:
        df_etapas["dataEtapaFim"] = pd.to_datetime(df_etapas["dataEtapaFim"], errors="coerce")
    else:
        df_etapas["dataEtapaFim"] = pd.NaT

    # ============================================================
    # 2. Identificar etapa(s) atual(is) por OS
    # CORREÇÃO ESP-DRIVE: preservar TODAS as etapas abertas.
    # Para OS sem etapa aberta (finalizadas): fallback = última etapa fechada (1 linha).
    # ============================================================
    df_etapas["etapa_aberta"] = df_etapas["dataEtapaFim"].isna()

    # Todas as etapas abertas (pode ser N por OS)
    df_etapas_abertas = df_etapas[df_etapas["etapa_aberta"]].copy()

    # OS sem nenhuma etapa aberta → fallback: última etapa fechada
    os_com_etapa_aberta = set(df_etapas_abertas["seqFluxo"])
    df_fallback = (
        df_etapas[~df_etapas["seqFluxo"].isin(os_com_etapa_aberta)]
        .sort_values(["seqFluxo", "dataAtenderEtapa"], ascending=[True, False])
        .drop_duplicates(subset="seqFluxo", keep="first")
    )

    df_etapa_atual = pd.concat([df_etapas_abertas, df_fallback], ignore_index=True)

    # Flag de observabilidade: 1 quando OS tem múltiplas etapas abertas
    contagem = (
        df_etapas_abertas
        .groupby("seqFluxo")
        .size()
        .reset_index(name="qtd_etapas_abertas")
    )
    df_etapa_atual = df_etapa_atual.merge(contagem, on="seqFluxo", how="left")
    df_etapa_atual["qtd_etapas_abertas"]    = df_etapa_atual["qtd_etapas_abertas"].fillna(1).astype(int)
    df_etapa_atual["flag_multiplas_etapas"] = (df_etapa_atual["qtd_etapas_abertas"] > 1).astype(int)

    os_multiplas = df_etapa_atual[df_etapa_atual["flag_multiplas_etapas"] == 1]["seqFluxo"].nunique()
    print(f"   OS com múltiplas etapas abertas: {os_multiplas}")
    print(f"   Total linhas de etapa geradas:   {len(df_etapa_atual):,}")

    # ============================================================
    # 3. Seleção e padronização de colunas
    # ============================================================
    colunas = ["seqFluxo", "etapa", "executor", "flag_multiplas_etapas"]
    servico_col = next(
        (c for c in ["servico", "Serviço", "nomeServico"] if c in df_etapa_atual.columns),
        None
    )
    if servico_col:
        colunas.append(servico_col)

    df_etapa_atual = df_etapa_atual[colunas].copy()

    rename_map = {"etapa": "etapa_atual", "executor": "executor_atual"}
    if servico_col and servico_col != "servico":
        rename_map[servico_col] = "servico"

    df_etapa_atual = df_etapa_atual.rename(columns=rename_map)

    # ============================================================
    # 4. Padronização da base de solicitações
    # ============================================================
    df = df_solicitacoes.copy()

    if "seqFluxo" not in df.columns:
        if "Nº Solicitação" in df.columns:
            df = df.rename(columns={"Nº Solicitação": "seqFluxo"})
        else:
            raise KeyError("Coluna seqFluxo não encontrada")

    df["seqFluxo"] = df["seqFluxo"].astype("int64")

    if "Data Criação" in df.columns:
        df["Data Criação"] = pd.to_datetime(df["Data Criação"], errors="coerce")

    # ============================================================
    # 5. Merge final
    # ============================================================
    df = df.merge(df_etapa_atual, on="seqFluxo", how="left")

    # OS sem etapa no Silver ficam com flag nulo após o merge — corrigir para 0
    df["flag_multiplas_etapas"] = df["flag_multiplas_etapas"].fillna(0).astype(int)

    if "servico" in df.columns and "Serviço" in df.columns:
        df["Serviço"] = df["servico"].fillna(df["Serviço"])
        df = df.drop(columns=["servico"])

    return df


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 10, Finished, Available, Finished, False)

In [7]:
#VALIDAÇÃO TESTE

df_teste = adicionar_etapa_atual_2(df_etapas, df_solicitacoes_tratado)

df_teste.loc[df_teste["seqFluxo"] == 669605, [
    "seqFluxo",
    "etapa_atual",
    "executor_atual"
]]


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 11, Finished, Available, Finished, False)

   OS com múltiplas etapas abertas: 252
   Total linhas de etapa geradas:   11,302


,seqFluxo,etapa_atual,executor_atual
5240,669605,FINALIZAR FLUXO,ADMINISTRAACTO ADMINISTRAACTO


In [8]:
df = adicionar_etapa_atual_2(
    df_etapas=df_obras_etapas,
    df_solicitacoes=df
)

print("✓ Etapa atual adicionada com sucesso")
print(f"✓ Registros: {len(df):,}")
print(f"✓ Colunas: {len(df.columns)}")


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 12, Finished, Available, Finished, False)

   OS com múltiplas etapas abertas: 252
   Total linhas de etapa geradas:   11,302
✓ Etapa atual adicionada com sucesso
✓ Registros: 11,303
✓ Colunas: 13


# Bairros

In [9]:
import pandas as pd

df = df.copy()

# ============================================================
# 1. Preservar valor original
# ============================================================

df["bairro_raw"] = df["Bairro"]

# ============================================================
# 2. Padronização básica (chave técnica)
# ============================================================

df["bairro_pad"] = (
    df["Bairro"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"[.,;:/\-]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# ============================================================
# 3. Correções conhecidas (acentuação e grafia)
# ============================================================

mapa_bairros = {
    # Acentuação
    "BOQUEIRAO": "BOQUEIRÃO",
    "POMPEIA": "POMPÉIA",
    "MARAPE": "MARAPÉ",
    "EMBARE": "EMBARÉ",
    "ESTUARIO": "ESTUÁRIO",
    "PAQUETA": "PAQUETÁ",
    "SABOO": "SABOÓ",
    "ITARARE": "ITARARÉ",

    # Variações comuns
    "PONTA PRAIA": "PONTA DA PRAIA",
    "PORTO PONTA DA PRAIA": "PONTA DA PRAIA",
    "PORTO PDAPRAIA": "PONTA DA PRAIA",

    # Vila Mathias
    "VILA MATIAS": "VILA MATHIAS",

    # Rádio Clube
    "RADIO CLUB": "RÁDIO CLUBE",
    "RADIO CLUBE": "RÁDIO CLUBE",
    "RÁDIO CLUBE I": "RÁDIO CLUBE",

    # Unificações frequentes
    "JOSE MENINO": "JOSÉ MENINO",
    "MARACANA": "MARACANÃ",
    "VILA FATIMA": "VILA FÁTIMA",
    "VL NOVA CONCEIÇÃO": "VILA NOVA CONCEIÇÃO",
    "MOR SÃO BENTO": "MORRO SÃO BENTO",
    "CHICO PAUL": "CHICO DE PAULA",
    "CHICO PAULA": "CHICO DE PAULA",
}

df["bairro_pad"] = df["bairro_pad"].replace(mapa_bairros)

# ============================================================
# 4. Normalização de morros
# ============================================================

df["bairro_pad"] = df["bairro_pad"].str.replace(
    r"^(MR|MR\.|MOR\.)\s*",
    "MORRO ",
    regex=True
)

# ============================================================
# 5. Remover valores inválidos óbvios
# ============================================================

valores_invalidos = {
    "NA", "NAN", "NONE",  # None/NaN em string vira "NONE" após upper
    "TESTE", "TESTEEE",
    "FDFD", "FDFDFDFDFDF",
    "AP 11", "BL 01 APTO 32",
    "KM 259"
}

df.loc[df["bairro_pad"].isin(valores_invalidos), "bairro_pad"] = pd.NA

# ============================================================
# 6. Marcar registros fora do município
# ============================================================

fora_municipio = {
    "ASA NORTE", "BROOKLIN", "BUTANTA", "MOEMA", "LIBERDADE",
    "PERDIZES", "TATUAPÉ", "PENHA DE FRANÇA", "PARQUE DA MOOCA",
    "BAL ESMERALDA", "PARADA INGLESA", "PARADA INGLESA SÃO PAULO"
}

df.loc[df["bairro_pad"].isin(fora_municipio), "bairro_pad"] = "FORA DO MUNICÍPIO"

print("✓ Padronização de bairros concluída")

# ============================================================
# 7. Conferência final
# ============================================================

bairros_final = (
    df["bairro_pad"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

display(bairros_final)
print(f"✓ Bairros padronizados válidos: {len(bairros_final)}")


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 13, Finished, Available, Finished, False)

✓ Padronização de bairros concluída


SynapseWidget(Synapse.DataFrame, 380db166-6201-4553-b9ee-4c6645769924)

✓ Bairros padronizados válidos: 158


In [10]:
# ============================================================
# 1. Carregar tabela auxiliar Zona_Bairros
# ============================================================

aux_path = "/lakehouse/default/Files/acto/PMS_AuxiliarPDR.xlsx"
aux_zona_bairros_pd = pd.read_excel(aux_path, sheet_name="Zona_Bairros")

aux_zona_bairros_pd = aux_zona_bairros_pd.dropna(subset=["BAIRRO"])
aux_zona_bairros_pd["BAIRRO"] = (
    aux_zona_bairros_pd["BAIRRO"]
    .astype(str)
    .str.upper()
    .str.strip()
)

if "ZONA" in aux_zona_bairros_pd.columns:
    aux_zona_bairros_pd["zona"] = aux_zona_bairros_pd["ZONA"].astype(str).str.strip()
else:
    aux_zona_bairros_pd["zona"] = ""

print(f"✓ Tabela auxiliar Zona_Bairros carregada: {len(aux_zona_bairros_pd):,} registros")

# ============================================================
# 1b. Carregar tabela auxiliar Etapas
# ============================================================

aux_etapas_pd = pd.read_excel(aux_path, sheet_name="Etapas")

aux_etapas_pd["etapa_aux"] = (
    aux_etapas_pd["Etapa"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

if "AuxSetorResponsável" in aux_etapas_pd.columns:
    aux_etapas_pd["aux_setor_responsavel"] = aux_etapas_pd["AuxSetorResponsável"]
else:
    aux_etapas_pd["aux_setor_responsavel"] = ""

aux_etapas = (
    aux_etapas_pd[["etapa_aux", "aux_setor_responsavel"]]
    .drop_duplicates(subset=["etapa_aux"], keep="first")
)

print(f"✓ Tabela auxiliar Etapas carregada: {len(aux_etapas):,} etapas únicas")

# ============================================================
# 2. Merge por etapa (aux_setor_responsavel)
# ============================================================

df["etapa_pad"] = (
    df["etapa_atual"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

df = (
    df.merge(
        aux_etapas,
        left_on="etapa_pad",
        right_on="etapa_aux",
        how="left"
    )
    .drop(columns=["etapa_aux", "etapa_pad"])
)

print("✓ Merge por etapa concluído (aux_setor_responsavel)")

# ============================================================
# 2b. Merge com df_obras_etapas (datas da etapa)
# ============================================================

df_obras_etapas["etapa_pad"] = (
    df_obras_etapas["etapa"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

for col in ["dataEtapaInicio", "dataEtapaFim"]:
    if col in df_obras_etapas.columns:
        df_obras_etapas[col] = (
            pd.to_datetime(df_obras_etapas[col], errors="coerce")
            .dt.tz_localize(None)
        )

df_etapas_merge = (
    df_obras_etapas[
        ["seqFluxo", "etapa_pad", "dataEtapaInicio", "dataEtapaFim", "tempoExecucao"]
    ]
    .sort_values("dataEtapaInicio", ascending=False)
    .drop_duplicates(subset=["seqFluxo", "etapa_pad"], keep="first")
)

df = (
    df.merge(
        df_etapas_merge,
        left_on=["seqFluxo", "etapa_atual"],
        right_on=["seqFluxo", "etapa_pad"],
        how="left"
    )
    .drop(columns=["etapa_pad"], errors="ignore")
)

print("✓ Merge com df_obras_etapas concluído")

# ============================================================
# 2c. Cálculo dias_na_etapa (ANTES da padronização)
# ============================================================

hoje = pd.Timestamp.now().tz_localize(None)

df["dias_na_etapa"] = pd.NA

# Status reais do sistema: "Em atendimento", "Cancelado", "Finalizado"
STATUS_ATIVO = {"EM ATENDIMENTO", "CANCELADO"}

mask_ativa = (
    df["Status Fluxo"]
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(STATUS_ATIVO)
    & df["dataEtapaInicio"].notna()
)

mask_finalizada = (
    df["Status Fluxo"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("FINALIZADO")
    & df["dataEtapaInicio"].notna()
    & df["dataEtapaFim"].notna()
)

# OS ativas: hoje - inicio da etapa
df.loc[mask_ativa, "dias_na_etapa"] = (
    hoje - df.loc[mask_ativa, "dataEtapaInicio"]
).dt.days

# OS finalizadas: fim da etapa - inicio da etapa (nao cresce com o tempo)
df.loc[mask_finalizada, "dias_na_etapa"] = (
    df.loc[mask_finalizada, "dataEtapaFim"]
    - df.loc[mask_finalizada, "dataEtapaInicio"]
).dt.days

print("✓ dias_na_etapa calculado")
print("  Ativas (hoje - inicio):      " + str(mask_ativa.sum()))
print("  Finalizadas (fim - inicio):  " + str(mask_finalizada.sum()))
print("  Sem calculo (sem data):      " + str(df["dias_na_etapa"].isna().sum()))

# ============================================================
# 3. Preparar auxiliar de bairros
# ============================================================

mapa_bairros = {
    "BOQUEIRAO": "BOQUEIRÃO",
    "POMPEIA": "POMPÉIA",
    "MARAPE": "MARAPÉ",
    "EMBARE": "EMBARÉ",
    "ESTUARIO": "ESTUÁRIO",
    "PAQUETA": "PAQUETÁ",
    "SABOO": "SABOÓ",
    "ITARARE": "ITARARÉ",
    "PONTA PRAIA": "PONTA DA PRAIA",
    "PORTO PONTA DA PRAIA": "PONTA DA PRAIA",
    "PORTO PDAPRAIA": "PONTA DA PRAIA",
    "VILA MATIAS": "VILA MATHIAS",
    "RADIO CLUB": "RÁDIO CLUBE",
    "RADIO CLUBE": "RÁDIO CLUBE",
    "RÁDIO CLUBE I": "RÁDIO CLUBE",
    "JOSE MENINO": "JOSÉ MENINO",
    "MARACANA": "MARACANÃ",
    "VILA FATIMA": "VILA FÁTIMA",
    "VL NOVA CONCEIÇÃO": "VILA NOVA CONCEIÇÃO",
    "MOR SÃO BENTO": "MORRO SÃO BENTO",
    "CHICO PAUL": "CHICO DE PAULA",
    "CHICO PAULA": "CHICO DE PAULA",
}

aux_zona_bairros_pd["bairro_nome"] = aux_zona_bairros_pd["BAIRRO"].astype(str).str.strip()

aux_zona_bairros_pd["bairro_pad"] = (
    aux_zona_bairros_pd["BAIRRO"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"[.,;:/\-]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .replace(mapa_bairros)
    .str.replace(r"^(MR|MR\.|MOR\.)\s*", "MORRO ", regex=True)
)

aux_zona_bairros = (
    aux_zona_bairros_pd[["bairro_pad", "zona", "bairro_nome"]]
    .drop_duplicates(subset=["bairro_pad"], keep="first")
)

print(f"✓ Auxiliar preparado: {len(aux_zona_bairros):,} bairros únicos")

# ============================================================
# 4. Merge por bairro
# ============================================================

df = df.merge(aux_zona_bairros, on="bairro_pad", how="left", validate="m:1")
print("✓ Merge por bairro concluído")

# ============================================================
# 5. bairro_consolidado
# ============================================================

df["bairro_consolidado"] = df["bairro_nome"].fillna(df["bairro_pad"])
df.loc[df["bairro_consolidado"].isin(["NONE", ""]), "bairro_consolidado"] = pd.NA

print("✓ bairro_consolidado criado")

# ============================================================
# 6. Limpeza intermediária
# ============================================================

colunas_para_remover = ["bairro_raw", "Bairro", "bairro_pad", "bairro_nome", "bairro"]
df = df.drop(columns=[c for c in colunas_para_remover if c in df.columns])

print("✓ Colunas intermediárias removidas")

# ============================================================
# 7. Padronização final (snake_case + datas)
# ============================================================

rename_cols = {
    "seqFluxo": "n_da_solicitacao",
    "Serviço": "servico",
    "Status Fluxo": "status",
    "Data Criação": "data_criacao",
    "Data Finalização": "data_finalizacao",
    "Solicitante": "solicitante",
    "Título Profissional": "titulo_profissional",
    "dataEtapaInicio":"data_etapa_inicio",
    "dataEtapaFim":"data_etapa_fim",
    "tempoExecucao":"tempo_execucao",
}

df = df.rename(columns={k: v for k, v in rename_cols.items() if k in df.columns})

for col in ["data_criacao", "data_finalizacao"]:
    if col in df.columns:
        s = pd.to_datetime(df[col], errors="coerce", utc=True)
        df[col] = s.dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ").where(s.notna(), pd.NA)

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].apply(lambda x: pd.NA if pd.isna(x) or x == "" else x)

print("✓ Pipeline finalizado com sucesso")


StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 14, Finished, Available, Finished, False)

✓ Tabela auxiliar Zona_Bairros carregada: 122 registros
✓ Tabela auxiliar Etapas carregada: 185 etapas únicas
✓ Merge por etapa concluído (aux_setor_responsavel)
✓ Merge com df_obras_etapas concluído
✓ dias_na_etapa calculado
  Ativas (hoje - inicio):      4053
  Finalizadas (fim - inicio):  6968
  Sem calculo (sem data):      282
✓ Auxiliar preparado: 100 bairros únicos
✓ Merge por bairro concluído
✓ bairro_consolidado criado
✓ Colunas intermediárias removidas
✓ Pipeline finalizado com sucesso


In [11]:
if "etapa_atual" in df.columns:
    df["etapa_atual"] = df["etapa_atual"].astype(str).str.strip()

# Remover colunas artefato (99% nulas, fora do schema)
df = df.drop(columns=["Executor", "Etapa"], errors="ignore")

# Criar zona_aplicavel: 0 para serviços sem relevância geográfica
SERVICOS_SEM_ZONA = {
    "INSCRIÇÃO DE PROFISSIONAL (PESSOA FÍSICA)",
    "INSCRIÇÃO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)",
    "RENOVAÇÃO DE CADASTRO PROFISSIONAL PESSOA FÍSICA",
    "RENOVAÇÃO DE CADASTRO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)",
    "PROVIDÊNCIA",
}
if "servico" in df.columns:
    df["zona_aplicavel"] = (~df["servico"].isin(SERVICOS_SEM_ZONA)).astype(int)
else:
    df["zona_aplicavel"] = 1

print(f"✓ etapa_atual normalizado")
print(f"✓ Colunas artefato removidas (Executor, Etapa)")
print(f"✓ zona_aplicavel criado — serviços sem zona: {(df['zona_aplicavel'] == 0).sum():,}")

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 15, Finished, Available, Finished, False)

✓ etapa_atual normalizado
✓ Colunas artefato removidas (Executor, Etapa)
✓ zona_aplicavel criado — serviços sem zona: 2,905


In [12]:
from pyspark.sql import functions as F

print("💾 Convertendo para Spark e salvando gold_pdr_acompanhamentos_os...")

df_pdr_spark = spark.createDataFrame(df)

# Garantir que colunas de data (ISO 8601) sejam timestamp no Delta
colunas_data = ["data_criacao", "data_finalizacao"]
for col in colunas_data:
    if col in df_pdr_spark.columns and dict(df_pdr_spark.dtypes).get(col) == "string":
        df_pdr_spark = df_pdr_spark.withColumn(
            col,
            F.to_timestamp(F.col(col), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS'Z'")
        )

# Salvar no Lakehouse
(
    df_pdr_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_pdr_acompanhamentos_os")
)

print("✓ Tabela gold_pdr_acompanhamentos_os salva com sucesso (sem aux_pdr)")
print(f"  Registros: {len(df):,} | Colunas: {df.columns.tolist()}")

StatementMeta(, 96086ab9-6030-43d1-b2b3-f58087b2347f, 16, Finished, Available, Finished, False)

💾 Convertendo para Spark e salvando gold_pdr_acompanhamentos_os...
✓ Tabela gold_pdr_acompanhamentos_os salva com sucesso (sem aux_pdr)
  Registros: 11,303 | Colunas: ['n_da_solicitacao', 'servico', 'status', 'data_criacao', 'data_finalizacao', 'solicitante', 'titulo_profissional', 'etapa_atual', 'executor_atual', 'flag_multiplas_etapas', 'aux_setor_responsavel', 'data_etapa_inicio', 'data_etapa_fim', 'tempo_execucao', 'dias_na_etapa', 'zona', 'bairro_consolidado', 'zona_aplicavel']
